# Diabetic Retinopathy — Manual Evaluation Control Panel

This notebook **does not train models**. Evaluate one completed model run at a time and keep each evaluation's files separate. Supported models:

- `efficientnet` — unweighted EfficientNetB0 baseline
- `efficientnet_weighted` — class-weighted EfficientNetB0
- `densenet` — class-weighted DenseNet121
- `fusion` — initial unweighted EfficientNetB0 + DenseNet121 feature fusion

Use the official `validation` split for model selection. Keep `test` held out until the model choices are settled. Test evaluation requires explicit confirmation and repeated test evaluation on the same model/run is blocked by default. Comparisons must use saved evaluations from the same split and require explicit selections.

## 1. Environment setup

Run this once after restarting the `dr_tf` kernel. It configures the Windows Conda DLL path before TensorFlow is imported and enables GPU memory growth. This cell does not load or evaluate a model.

In [1]:
from configs import tensorflow_setup as _tensorflow_setup
import sys
import numpy as np
import tensorflow as tf
_tensorflow_setup.configure_gpu_memory_growth(tf)

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("EfficientNetB0 API:", hasattr(tf.keras.applications, "EfficientNetB0"))
print("DenseNet121 API:", hasattr(tf.keras.applications, "DenseNet121"))

from scripts.evaluate import (
    evaluate_model,
    list_saved_evaluations,
    compare_evaluations,
    MODEL_DISPLAY_NAMES,
)
from scripts.utils import print_run_status

print("Evaluation import successful. No model has been loaded yet.")

Python: 3.10.20
TensorFlow: 2.10.1
NumPy: 1.23.5
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
EfficientNetB0 API: True
DenseNet121 API: True
Evaluation import successful. No model has been loaded yet.


## 2. Inspect each model's runs

Check that the run you intend to evaluate is marked `completed` and has its best-validation weights. This cell only reads run metadata/status; it does not evaluate a model.

In [2]:
model_names = ["efficientnet", "efficientnet_weighted", "densenet", "fusion"]
for selected_model in model_names:
    print("\n" + "=" * 72)
    print_run_status(selected_model)


Model: efficientnet
run_001: status=completed, epoch=35/35, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet\runs\run_001

Model: efficientnet_weighted
run_001: status=completed, epoch=50/50, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet_weighted\runs\run_001

Model: densenet
run_001: status=completed, epoch=50/50, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\densenet\runs\run_001

Model: fusion
run_001: status=completed, epoch=40/40, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\fusion\runs\run_001


## 3. Unweighted EfficientNetB0 — evaluate one run

Enter a specific **completed** run ID such as `run_001`. Choose `validation` during model selection. Choose `test` only after the model choices are settled and type `FINAL-TEST` when prompted. Every call writes a new evaluation folder; existing results are never overwritten.

In [3]:
efficientnet_eval = None
model_name = "efficientnet"
run_id = input("Completed run ID for efficientnet (example run_001): ").strip()
split = input("Evaluation split [validation/test] (default validation): ").strip().lower() or "validation"
batch_text = input("Evaluation batch size [4]: ").strip()
batch_size = int(batch_text or 4)

if split == "test":
    final_test_choice = input("Test split is held out. Type FINAL-TEST to proceed: ").strip()
    if final_test_choice != "FINAL-TEST":
        print("Evaluation cancelled; test split was not evaluated.")
    else:
        efficientnet_eval = evaluate_model(
            model_name=model_name,
            run_id=run_id,
            split=split,
            batch_size=batch_size,
            confirm_test=True,
        )
elif split == "validation":
    efficientnet_eval = evaluate_model(
        model_name=model_name,
        run_id=run_id,
        split=split,
        batch_size=batch_size,
    )
else:
    raise ValueError("Split must be 'validation' or 'test'.")

Evaluated 4,000 examples...
Evaluated 8,000 examples...

Evaluation saved to: D:\M L\Diabetic_Retinopathy_Detection_mini_project\results\evaluations\efficientnet\run_001\eval_001
Split: validation; examples: 10,906
Accuracy: 0.7609
Macro-F1: 0.3095
Quadratic weighted kappa: 0.3540453391913625
Recall — No DR: 0.9926
Recall — Mild NPDR: 0.0000
Recall — Moderate NPDR: 0.0842
Recall — Severe NPDR: 0.0549
Recall — Proliferative DR (PDR): 0.3417


## 4. Class-weighted EfficientNetB0 — evaluate one run

Enter a specific **completed** run ID such as `run_001`. Choose `validation` during model selection. Choose `test` only after the model choices are settled and type `FINAL-TEST` when prompted. Every call writes a new evaluation folder; existing results are never overwritten.

In [4]:
weighted_efficientnet_eval = None
model_name = "efficientnet_weighted"
run_id = input("Completed run ID for efficientnet_weighted (example run_001): ").strip()
split = input("Evaluation split [validation/test] (default validation): ").strip().lower() or "validation"
batch_text = input("Evaluation batch size [4]: ").strip()
batch_size = int(batch_text or 4)

if split == "test":
    final_test_choice = input("Test split is held out. Type FINAL-TEST to proceed: ").strip()
    if final_test_choice != "FINAL-TEST":
        print("Evaluation cancelled; test split was not evaluated.")
    else:
        weighted_efficientnet_eval = evaluate_model(
            model_name=model_name,
            run_id=run_id,
            split=split,
            batch_size=batch_size,
            confirm_test=True,
        )
elif split == "validation":
    weighted_efficientnet_eval = evaluate_model(
        model_name=model_name,
        run_id=run_id,
        split=split,
        batch_size=batch_size,
    )
else:
    raise ValueError("Split must be 'validation' or 'test'.")

Evaluated 4,000 examples...
Evaluated 8,000 examples...

Evaluation saved to: D:\M L\Diabetic_Retinopathy_Detection_mini_project\results\evaluations\efficientnet_weighted\run_001\eval_001
Split: validation; examples: 10,906
Accuracy: 0.7331
Macro-F1: 0.3683
Quadratic weighted kappa: 0.4830979382422801
Recall — No DR: 0.8999
Recall — Mild NPDR: 0.0153
Recall — Moderate NPDR: 0.3065
Recall — Severe NPDR: 0.1603
Recall — Proliferative DR (PDR): 0.6083


## 5. Class-weighted DenseNet121 — evaluate one run

Enter a specific **completed** run ID such as `run_001`. Choose `validation` during model selection. Choose `test` only after the model choices are settled and type `FINAL-TEST` when prompted. Every call writes a new evaluation folder; existing results are never overwritten.

In [5]:
densenet_eval = None
model_name = "densenet"
run_id = input("Completed run ID for densenet (example run_001): ").strip()
split = input("Evaluation split [validation/test] (default validation): ").strip().lower() or "validation"
batch_text = input("Evaluation batch size [4]: ").strip()
batch_size = int(batch_text or 4)

if split == "test":
    final_test_choice = input("Test split is held out. Type FINAL-TEST to proceed: ").strip()
    if final_test_choice != "FINAL-TEST":
        print("Evaluation cancelled; test split was not evaluated.")
    else:
        densenet_eval = evaluate_model(
            model_name=model_name,
            run_id=run_id,
            split=split,
            batch_size=batch_size,
            confirm_test=True,
        )
elif split == "validation":
    densenet_eval = evaluate_model(
        model_name=model_name,
        run_id=run_id,
        split=split,
        batch_size=batch_size,
    )
else:
    raise ValueError("Split must be 'validation' or 'test'.")

Evaluated 1,000 examples...
Evaluated 2,000 examples...
Evaluated 3,000 examples...
Evaluated 4,000 examples...
Evaluated 5,000 examples...
Evaluated 6,000 examples...
Evaluated 7,000 examples...
Evaluated 8,000 examples...
Evaluated 9,000 examples...
Evaluated 10,000 examples...

Evaluation saved to: D:\M L\Diabetic_Retinopathy_Detection_mini_project\results\evaluations\densenet\run_001\eval_001
Split: validation; examples: 10,906
Accuracy: 0.7014
Macro-F1: 0.3052
Quadratic weighted kappa: 0.4445732949436494
Recall — No DR: 0.9023
Recall — Mild NPDR: 0.0333
Recall — Moderate NPDR: 0.0057
Recall — Severe NPDR: 0.6329
Recall — Proliferative DR (PDR): 0.5458


## 6. Feature-fusion model — evaluate one run

Enter a specific **completed** run ID such as `run_001`. Choose `validation` during model selection. Choose `test` only after the model choices are settled and type `FINAL-TEST` when prompted. Every call writes a new evaluation folder; existing results are never overwritten.

In [6]:
fusion_eval = None
model_name = "fusion"
run_id = input("Completed run ID for fusion (example run_001): ").strip()
split = input("Evaluation split [validation/test] (default validation): ").strip().lower() or "validation"
batch_text = input("Evaluation batch size [4]: ").strip()
batch_size = int(batch_text or 4)

if split == "test":
    final_test_choice = input("Test split is held out. Type FINAL-TEST to proceed: ").strip()
    if final_test_choice != "FINAL-TEST":
        print("Evaluation cancelled; test split was not evaluated.")
    else:
        fusion_eval = evaluate_model(
            model_name=model_name,
            run_id=run_id,
            split=split,
            batch_size=batch_size,
            confirm_test=True,
        )
elif split == "validation":
    fusion_eval = evaluate_model(
        model_name=model_name,
        run_id=run_id,
        split=split,
        batch_size=batch_size,
    )
else:
    raise ValueError("Split must be 'validation' or 'test'.")

Evaluated 1,000 examples...
Evaluated 2,000 examples...
Evaluated 3,000 examples...
Evaluated 4,000 examples...
Evaluated 5,000 examples...
Evaluated 6,000 examples...
Evaluated 7,000 examples...
Evaluated 8,000 examples...
Evaluated 9,000 examples...
Evaluated 10,000 examples...

Evaluation saved to: D:\M L\Diabetic_Retinopathy_Detection_mini_project\results\evaluations\fusion\run_001\eval_001
Split: validation; examples: 10,906
Accuracy: 0.7662
Macro-F1: 0.3504
Quadratic weighted kappa: 0.41515012978186416
Recall — No DR: 0.9883
Recall — Mild NPDR: 0.0000
Recall — Moderate NPDR: 0.1260
Recall — Severe NPDR: 0.1055
Recall — Proliferative DR (PDR): 0.4042


## 7. List saved evaluations

This lists existing reports only. It does not re-evaluate or rank saved models automatically.

In [7]:
import pandas as pd

saved_evaluations = list_saved_evaluations()
if saved_evaluations:
    display(pd.DataFrame(saved_evaluations))
else:
    print("No evaluation reports are saved yet.")

,model_name,model_display_name,run_id,evaluation_id,split,accuracy,macro_f1,weighted_f1,quadratic_weighted_kappa,output_dir
0,densenet,DenseNet121 (class-weighted),run_001,eval_001,validation,0.701449,0.305220,0.654182,0.444573,D:\M L\Diabetic_Retinopathy_Detection_mini_pro...
1,efficientnet,EfficientNetB0 (unweighted),run_001,eval_001,validation,0.760866,0.309485,0.680264,0.354045,D:\M L\Diabetic_Retinopathy_Detection_mini_pro...
2,efficientnet_weighted,EfficientNetB0 (class-weighted),run_001,eval_001,validation,0.733083,0.368269,0.703146,0.483098,D:\M L\Diabetic_Retinopathy_Detection_mini_pro...
3,fusion,EfficientNetB0 + DenseNet121 fusion (unweighted),run_001,eval_001,validation,0.766184,0.350435,0.694298,0.415150,D:\M L\Diabetic_Retinopathy_Detection_mini_pro...


## 8. Compare explicitly selected evaluations

Select at least two saved reports by entering their exact model key, run ID, and evaluation ID. The script refuses to compare a mixture of `validation` and `test` reports. It does not sweep through all saved runs; only the reports you enter are compared. The comparison adds per-class recall columns because minority-grade performance matters for this imbalanced dataset.

In [9]:
comparison_count = int(input("Number of saved evaluations to compare [4]: ").strip() or 4)
if comparison_count < 2:
    raise ValueError("Enter at least two evaluation reports to compare.")

selected_evaluations = []
for index in range(comparison_count):
    print(f"\nEvaluation {index + 1} of {comparison_count}")
    selected_evaluations.append({
        "model_name": input("Model key [efficientnet / efficientnet_weighted / densenet / fusion]: ").strip(),
        "run_id": input("Run ID (example run_001): ").strip(),
        "evaluation_id": input("Evaluation ID (example eval_001): ").strip(),
    })

comparison_result = compare_evaluations(selected_evaluations)
display(pd.DataFrame(comparison_result["rows"]))
print("Comparison files:", comparison_result["output_dir"])


Evaluation 1 of 4

Evaluation 2 of 4

Evaluation 3 of 4

Evaluation 4 of 4
Comparison saved to: D:\M L\Diabetic_Retinopathy_Detection_mini_project\results\comparisons\comparison_001


,model_name,model,run_id,evaluation_id,split,n_examples,accuracy,macro_f1,weighted_f1,quadratic_weighted_kappa,recall_no_dr,recall_mild_npdr,recall_moderate_npdr,recall_severe_npdr,recall_proliferative_dr_pdr
0,efficientnet,EfficientNetB0 (unweighted),run_001,eval_001,validation,10906,0.760866,0.309485,0.680264,0.354045,0.992620,0.000000,0.084231,0.054852,0.341667
1,efficientnet_weighted,EfficientNetB0 (class-weighted),run_001,eval_001,validation,10906,0.733083,0.368269,0.703146,0.483098,0.899877,0.015278,0.306523,0.160338,0.608333
2,densenet,DenseNet121 (class-weighted),run_001,eval_001,validation,10906,0.701449,0.305220,0.654182,0.444573,0.902337,0.033333,0.005700,0.632911,0.545833
3,fusion,EfficientNetB0 + DenseNet121 fusion (unweighted),run_001,eval_001,validation,10906,0.766184,0.350435,0.694298,0.415150,0.988315,0.000000,0.126029,0.105485,0.404167


Comparison files: D:\M L\Diabetic_Retinopathy_Detection_mini_project\results\comparisons\comparison_001


## Output organization

- Model-run metrics, class recalls, confusion matrix, and predictions: `results/evaluations/<model_name>/<run_id>/eval_NNN/`
- A comparison of only the selected evaluation reports: `results/comparisons/comparison_NNN/`

Metrics include accuracy, macro-F1, weighted-F1, class-wise precision/recall/F1/support, confusion matrix, and quadratic weighted kappa (QWK). `predictions.csv` contains the split's enumerated example index, true/predicted label, and five class probabilities. Dataset examples are not rebalanced during evaluation.